## 🎯 Learning Objectives
* Understand the fundamental concept of Policy Gradient methods in Reinforcement Learning.
* Grasp the core principles and mathematical intuition behind the REINFORCE algorithm.
* Implement the REINFORCE algorithm from scratch using PyTorch for a simple control task.
* Analyze the performance characteristics, strengths, and limitations of REINFORCE.
* Identify scenarios where REINFORCE or its derivatives are applicable.


## Policy Gradient Methods: REINFORCE

Welcome to the foundational lesson on Policy Gradient methods, a cornerstone of modern Reinforcement Learning. Unlike Value-Based methods (like Q-learning or SARSA) that learn the optimal value function and derive a policy from it, Policy Gradient methods directly optimize the policy function itself. This direct approach offers significant advantages, especially in environments with continuous action spaces or when dealing with stochastic policies.

### The Core Idea: Direct Policy Optimization

Imagine you're a chef trying to perfect a new recipe. Instead of meticulously calculating the 'value' of each ingredient combination (which would be incredibly complex), you simply try different ingredient ratios, taste the result, and adjust. If the dish tastes better, you slightly increase the proportions of the ingredients that led to that improvement. If it tastes worse, you reduce them. This iterative adjustment, directly based on the outcome, is analogous to Policy Gradient methods.

In RL terms, our 'recipe' is the policy $\pi(a|s; \theta)$, a parameterized function (often a neural network) that outputs the probability of taking action $a$ in state $s$, given parameters $\theta$. Our 'taste test' is the reward signal. We want to adjust $\theta$ such that the expected cumulative reward (return) is maximized.

Mathematically, we aim to maximize the expected return $J(\theta) = E_{\pi_{\theta}}[G_t]$, where $G_t$ is the discounted sum of future rewards from time $t$. To do this, we use gradient ascent:

$\theta_{new} = \theta_{old} + \alpha \nabla J(\theta)$

The challenge lies in computing $\nabla J(\theta)$. The Policy Gradient Theorem provides a way to do this, even when we don't know the environment's dynamics. For episodic tasks, it states:

$\nabla J(\theta) = E_{\pi_{\theta}}[\nabla \log \pi(A_t|S_t; \theta) G_t]$

Here, $\nabla \log \pi(A_t|S_t; \theta)$ is the gradient of the log-probability of the action taken, and $G_t$ is the return (total discounted reward) following that action. This is often called the 'score function' or 'log-derivative' trick.

### REINFORCE: The Monte Carlo Policy Gradient

REINFORCE (Reward Increment = Nonnegative Factor * Offset Reinforcement * Characteristic Eligibility) is the simplest and most foundational Policy Gradient algorithm. It's a Monte Carlo method because it relies on complete episodes to estimate the return $G_t$. 

**How REINFORCE Works (Step-by-Step):**

1.  **Initialize Policy Network:** Create a neural network with parameters $\theta$ that takes a state as input and outputs probabilities for each action.
2.  **Generate an Episode:** Interact with the environment using the current policy $\pi_{\theta}$ to collect a full trajectory: $S_0, A_0, R_1, S_1, A_1, R_2, ..., S_{T-1}, A_{T-1}, R_T, S_T$.
3.  **Calculate Returns:** For each time step $t$ in the episode, calculate the discounted return $G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... + \gamma^{T-t-1} R_T$. This is the total future reward from that point onwards.
4.  **Compute Policy Gradient:** For each time step $t$, compute the gradient term $\nabla \log \pi(A_t|S_t; \theta) G_t$. The $G_t$ term acts as a 'weight' for the action's log-probability gradient. If $G_t$ is positive, we want to increase the probability of taking $A_t$ in $S_t$. If $G_t$ is negative, we want to decrease it.
5.  **Update Policy Parameters:** Sum up all the gradient terms from the episode and perform a gradient ascent step:
    $\theta \leftarrow \theta + \alpha \sum_{t=0}^{T-1} \nabla \log \pi(A_t|S_t; \theta) G_t$
6.  **Repeat:** Go back to step 2 and generate a new episode, continuously refining the policy.

### Intuition Behind $G_t$ as a Weight

The $G_t$ term is crucial. It tells us how 'good' the action $A_t$ taken in state $S_t$ was, in terms of future rewards. If $G_t$ is high, it means that action led to a favorable outcome, so we want to increase its probability. If $G_t$ is low (or negative), it means the action led to a poor outcome, so we want to decrease its probability. The log-probability gradient ensures that the update moves the policy parameters in the direction that increases the probability of 'good' actions and decreases the probability of 'bad' actions.

REINFORCE is simple and elegant, but it has a significant drawback: high variance. Because it uses the full Monte Carlo return $G_t$ from a single episode, the estimate of the gradient can be very noisy. This often leads to slow and unstable learning. However, it forms the basis for more advanced and stable policy gradient methods like Actor-Critic, A2C, A3C, and PPO, which we will explore in subsequent lessons.


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt

# Ensure reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Define the Policy Network
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, action_dim)
        # Softmax to output probabilities for actions
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, state):
        x = self.fc1(state)
        x = self.relu(x)
        action_probs = self.softmax(self.fc2(x))
        return action_probs

    def act(self, state):
        # Convert state to tensor and add batch dimension
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        action_probs = self.forward(state_tensor)
        # Create a categorical distribution to sample an action
        m = Categorical(action_probs)
        action = m.sample()
        # Return the action and its log-probability
        return action.item(), m.log_prob(action)

# 2. REINFORCE Algorithm Implementation
def reinforce(env_name, n_episodes=1000, gamma=0.99, lr=1e-2):
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy_net = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)

    episode_rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset(seed=SEED + episode) # Reset environment for a new episode
        log_probs = []
        rewards = []

        done = False
        while not done:
            action, log_prob = policy_net.act(state)
            next_state, reward, done, truncated, _ = env.step(action)

            log_probs.append(log_prob) # Store log-probability of the action taken
            rewards.append(reward)     # Store the reward received
            state = next_state

            if done or truncated:
                break

        episode_rewards.append(sum(rewards))

        # Calculate discounted returns (G_t)
        returns = []
        G = 0
        # Iterate backwards through rewards to calculate discounted returns
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G) # Insert at the beginning to maintain original order

        # Convert returns to tensor
        returns = torch.tensor(returns)
        # Optional: Normalize returns for better stability (common practice)
        # returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        # Calculate the policy loss
        # The loss is -log_prob * G_t. We want to maximize expected reward, so we minimize -expected_reward.
        # Sum over all time steps in the episode
        policy_loss = []
        for log_prob, G_t in zip(log_probs, returns):
            policy_loss.append(-log_prob * G_t)

        # Backpropagation
        optimizer.zero_grad()
        # Sum all individual losses for the episode and backpropagate
        torch.stack(policy_loss).sum().backward()
        optimizer.step()

        if (episode + 1) % 100 == 0:
            print(f"Episode {episode + 1}/{n_episodes}, Average Reward: {np.mean(episode_rewards[-100:]):.2f}")
        
        # Check for convergence (CartPole is solved if avg reward over 100 episodes is >= 195)
        if np.mean(episode_rewards[-100:]) >= 195 and len(episode_rewards) >= 100:
            print(f"Environment solved in {episode + 1} episodes!")
            break

    env.close()
    return episode_rewards

# 3. Run the REINFORCE algorithm on CartPole-v1
print("Starting REINFORCE training on CartPole-v1...")
rewards = reinforce('CartPole-v1', n_episodes=2000)

# 4. Plotting the results
plt.figure(figsize=(12, 6))
plt.plot(rewards)
plt.title('REINFORCE: Rewards per Episode on CartPole-v1')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.grid(True)
plt.show()

# Plotting moving average for smoother visualization
window_size = 100
moving_avg = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
plt.figure(figsize=(12, 6))
plt.plot(moving_avg)
plt.title(f'REINFORCE: Moving Average of Rewards ({window_size}-episode window)')
plt.xlabel('Episode')
plt.ylabel('Average Total Reward')
plt.grid(True)
plt.show()


### Interpreting the Code Output and Performance Trade-offs

When you run the provided code, you'll observe the REINFORCE algorithm learning to balance the CartPole. The `print` statements will show the average reward over the last 100 episodes. Initially, this average reward will be low (around 20-30), indicating random or poor performance. As training progresses, you should see this average reward steadily increase, eventually reaching and ideally surpassing the `195` threshold, which signifies that the CartPole environment is considered 'solved'. The plots will visually confirm this learning trend, with the moving average plot providing a smoother representation of the policy's improvement.

#### How to Interpret the Learning Curve:

*   **Initial Phase (Low Rewards):** The agent's policy is essentially random. The pole falls quickly, resulting in short episodes and low rewards.
*   **Learning Phase (Increasing Rewards):** The policy network starts to identify actions that lead to higher returns. The `G_t` term guides the updates, reinforcing 'good' actions. You'll see the average reward gradually climb.
*   **Convergence (High Rewards):** The policy becomes robust, consistently balancing the pole for extended periods, leading to high episode rewards (up to 200 for CartPole-v1).

#### Performance Trade-offs of REINFORCE:

**Strengths:**

1.  **Simplicity and Foundation:** REINFORCE is conceptually straightforward and forms the bedrock for understanding more complex policy gradient methods. Its direct policy optimization is elegant.
2.  **Model-Free:** It doesn't require knowledge of the environment's dynamics, making it applicable to a wide range of real-world problems where a model is unavailable or too complex.
3.  **Handles Continuous Action Spaces:** With appropriate policy network architectures (e.g., outputting parameters for a Gaussian distribution), REINFORCE can naturally handle continuous action spaces, a significant advantage over many value-based methods.
4.  **Stochastic Policies:** It can learn stochastic policies, which are often beneficial in environments with partial observability or when exploration is crucial.

**Weaknesses:**

1.  **High Variance:** This is REINFORCE's most significant drawback. Because it uses Monte Carlo estimates of the return ($G_t$) from a single, complete episode, the gradient estimates can be very noisy. This high variance leads to slow convergence and can make training unstable.
2.  **Credit Assignment Problem:** While $G_t$ assigns credit, it assigns it to *all* actions taken in an episode. An action taken early in a long episode might receive credit (or blame) for rewards that are only realized much later, even if intervening actions were more influential. This makes learning less efficient.
3.  **Sensitivity to Hyperparameters:** Learning rate, discount factor, and network architecture can significantly impact performance, requiring careful tuning.
4.  **No Baseline:** The basic REINFORCE algorithm lacks a baseline, meaning $G_t$ can be large and positive even for relatively poor episodes, or small and negative for relatively good ones, simply due to the inherent stochasticity of the environment. This exacerbates the variance issue. (Note: The code includes an optional commented-out line for normalizing returns, which acts as a simple baseline).

#### Typical Use Cases:

*   **Educational Tool:** Excellent for understanding the fundamentals of policy gradient methods before diving into more complex algorithms.
*   **Simple Control Tasks:** For environments with relatively short episodes and clear reward signals where the variance isn't prohibitive.
*   **Baseline for Research:** Often used as a baseline to compare the performance of novel policy gradient algorithms.
*   **Foundation for Advanced Methods:** The core idea of directly optimizing a policy and using the log-probability trick is fundamental to algorithms like Actor-Critic, A2C, A3C, TRPO, and PPO, which address REINFORCE's variance issues through various techniques (e.g., using a critic, generalized advantage estimation, trust regions, or clipped objectives).


### Resources for Further Learning

To deepen your understanding of Policy Gradient methods and REINFORCE, explore the following resources:

*   **Sutton & Barto - Reinforcement Learning: An Introduction (2nd Edition):**
    *   Chapter 13: Policy Gradient Methods. This is the definitive theoretical reference for REINFORCE and its derivatives. [Link to online version](http://incompleteideas.net/book/RLbook2020.pdf)

*   **PyTorch Documentation:**
    *   `torch.nn.Linear`: [Link](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
    *   `torch.nn.ReLU`: [Link](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
    *   `torch.nn.Softmax`: [Link](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
    *   `torch.optim.Adam`: [Link](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)
    *   `torch.distributions.Categorical`: Essential for sampling actions from a probability distribution and getting log-probabilities. [Link](https://pytorch.org/docs/stable/distributions.html#categorical)

*   **Gymnasium Documentation:**
    *   Official documentation for the environment library used: [Link](https://gymnasium.farama.org/)

*   **Hugging Face TRL (Transformer Reinforcement Learning) Library:**
    *   While REINFORCE is foundational, modern RL for LLMs often uses more advanced techniques. TRL provides tools for training language models with RL, including Proximal Policy Optimization (PPO), which builds upon policy gradient concepts. Understanding REINFORCE is a crucial step towards understanding PPO and RLHF. [Link](https://huggingface.co/docs/trl/index)

*   **Google AI Blog / DeepMind Publications:**
    *   Search for articles on "Policy Gradients" or "REINFORCE" for intuitive explanations and historical context. These often provide excellent high-level overviews and practical insights.
